In [1]:
import math
import numpy as np 
import netCDF4 as nc
import xarray as xr
import thermofeel as tmf
import matplotlib.pyplot as plt
from scipy import ndimage

import zipfile
import os
import shutil
import gc
import cdsapi
from datetime import datetime

### define some functions that will be useful later

In [2]:
################################################################################
######################### FUNCTIONS TO CREATE A RELEVENT API REQUEST
##################################################################################

def number_of_days_in_month(month, year):
    month = int(month)
    year = int(year)
    
    # Days in each month
    if month in [1, 3, 5, 7, 8, 10, 12]:
        n_days = 31
    elif month in [4, 6, 9, 11]:
        n_days = 30
    else:  # February
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            n_days = 29
        else:
            n_days = 28
    
    # Return list of zero-padded strings
    return [f"{i:02d}" for i in range(1, n_days + 1)]


def needed_months(year):
    year = int(year)

    #define the needed months depending on the year. 
    # For 2014 and 2018, we only need data from October to Decembre

    if year in [2014, 2018]:
        start_month = 10
        end_month = 12
    # For other years, we need data from all months
    else : 
        start_month = 1
        end_month = 12

    return [f"{i:02d}" for i in range (start_month, end_month + 1)]


################################################################################
######################### DOWNLOADING FUNCTIONS
##################################################################################

def download_REAN_data(y, m, days, A, path):
    dataset = "reanalysis-era5-land"
    request = {
        "product_type": ["reanalysis"],
        "variable": [
            "2m_temperature",
            "total_precipitation"
        ],
        "year": y,
        "month": m,
        "day": days,
        "time": [f"{h:02d}:00" for h in range(24)],
        "data_format": "netcdf",
        "download_format": "zip",
        "area": A
    }
    
    c= cdsapi.Client()
    c.retrieve(dataset, request, path)



################################################################################
######################### FUNCTION TO DELETE THE TEMPORARY FILES
##################################################################################


def delete_files(folder):
    for filename in os.listdir(folder):
        file_path = os.path.join(folder, filename)
        try:
            if os.path.isfile(file_path) or os.path.islink(file_path):
                os.remove(file_path)  # remove file or symlink
            elif os.path.isdir(file_path):
                shutil.rmtree(file_path)  # remove subfolder
        except Exception as e:
            print(f'Failed to delete {file_path}. Reason: {e}')
    print(f"Deleted all contents of folder: {folder}")

### Enter the parameters that are needed to scrap the ERA5 data

In [3]:
# the folder in which the zip files are downnloaded
download_folder = 'interm/zip_files/010_files'
#the folder in which the data are saved
data_folder = 'output/'


#the years needed
years = ['2014', '2015', '2016',  '2018', '2019']

#the coordinates needed. Carreful : the coordinates should be rounded at the (upper or lower) 0.10. For instance : 35.55->35.5
India_area = [35.5, 68.0, 6.5, 97.5]

### The following loop creates netcdf files containing hourly climate data. 
#### If the zip files already exist in the 'interm' folder, you do need to download them again. Count roughly 5 to 7 seconds per go in the loop.
#### If the zip files do not exist in the 'interm' folder, the script downloads them using the ERA5 CDS API. Count roughly 7minutes per go in the loop.

In [4]:
start = datetime.now()
for y in years:
    #get the required months
    months = needed_months(y)
    
    for m in months:
        start1 = datetime.now()
        #get the days corresponding to month m and year y
        needed_days = number_of_days_in_month(m, y)
    
        ################################################################# DOWNLOAD THE REANALYSIS DATA
        #check if the data already exist for this year-month. if no, download them.
        filename = 'REAN_' + y + m + '.zip'
        filepath = os.path.join(download_folder, filename)
        if os.path.exists(filepath):
            print(f"File {filename} already exists")
        else:
            print(f"File {filename} does not exist in the demanded folder. Proceed to download.")
            download_REAN_data(y, m, needed_days, India_area, filepath)
            print(f"File {filename} downloaded")

        
        
        ########################################################### UNZIP, OPEN AND STORE THE NEEDED DATA INTO A UNIQUE XRARRAY OBJECT called REAN
        
        #Path to reanalysis zipped archive
        zip_path = os.path.join(download_folder, filename)
        extract_dir =  'interm/unzipped_REAN'
        
        #unzip the REAN file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        
        #collect the two netcdf from the REAN unzipped archive
        REAN = xr.open_dataset(os.path.join(extract_dir, 'data_0.nc'), engine = "h5netcdf").load()
        REAN.close()

        #rename the REAN dimensions
        REAN = REAN.rename({
            "valid_time": "time",
            "latitude": "lat",
            "longitude": "lon"
        })
    
        ################################################################# SAVE THE FINAL NETCDF FOR THE CURRENT MONTH
    
        vars_to_keep = ['t2m', 'tp']
        subset = REAN[vars_to_keep]
        
        output_path = os.path.join(data_folder, '010_alltemps_' + y + m + '.nc')
        subset.to_netcdf(output_path)

        print('File ' + '010_alltemps_' + y + m + '.nc' + ' saved')    

        
        ################################################################# CLEAN EVERYTHING FOR THE NEXT STEP
        
        #empty the extract folders
        extract_dir = ['interm/unzipped_MRT', 'interm/unzipped_REAN']
        for folder in extract_dir:
            delete_files(folder)
        end = datetime.now()
        print(f'this stage took {end-start1}')
        print(f'total time since entering the loop: {end-start}')
        print('-------------------------------------------------')

File REAN_201410.zip already exists
File 010_alltemps_201410.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:08.581742
total time since entering the loop: 0:00:08.581967
-------------------------------------------------
File REAN_201411.zip already exists
File 010_alltemps_201411.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:04.487750
total time since entering the loop: 0:00:13.069743
-------------------------------------------------
File REAN_201412.zip already exists
File 010_alltemps_201412.nc saved
Deleted all contents of folder: interm/unzipped_MRT
Deleted all contents of folder: interm/unzipped_REAN
this stage took 0:00:04.477901
total time since entering the loop: 0:00:17.547686
-------------------------------------------------
File REAN_201501.zip does not exist in the demanded folder. Proceed to download

2026-04-20 11:58:54,355 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-04-20 11:58:54,355 INFO Request ID is 7a8d7dfc-800c-4d10-8630-e8cd504ca583
2026-04-20 11:58:54,412 INFO status has been updated to accepted
2026-04-20 11:59:08,170 INFO status has been updated to running


KeyboardInterrupt: 